In [1]:
from pathlib import Path
import duckdb

DB_PATH = "trips.duckdb"
OUTPUT_DIR = Path("outputs/csv_exports")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect(DB_PATH, read_only=True)

# Export core marts
con.execute(f"COPY fct_daily_stats TO '{(OUTPUT_DIR / 'fct_daily_stats.csv').as_posix()}' (HEADER, DELIMITER ',')")
con.execute(f"COPY dim_zones TO '{(OUTPUT_DIR / 'dim_zones.csv').as_posix()}' (HEADER, DELIMITER ',')")
con.execute(f"COPY dim_time TO '{(OUTPUT_DIR / 'dim_time.csv').as_posix()}' (HEADER, DELIMITER ',')")

con.execute(f"""
    COPY (
        SELECT *
        FROM fct_trips
        ORDER BY pickup_datetime
        LIMIT 100000
    )
    TO '{(OUTPUT_DIR / 'fct_trips_sample.csv').as_posix()}'
    (HEADER, DELIMITER ',')
""")

print("CSV files exported to:", OUTPUT_DIR.resolve())

# con.execute(f"""
#     COPY fct_trips
#     TO '{(OUTPUT_DIR / 'fct_trips_full.csv').as_posix()}'
#     (HEADER, DELIMITER ',')
# """)

# print("Full fact table exported.")

con.execute(f"""
COPY (
    SELECT
        trip_date                    AS "Trip Date",
        trip_year                    AS "Trip Year",
        trip_month                   AS "Trip Month",
        day_name                     AS "Day Name",
        is_weekend                   AS "Is Weekend",
        service_type                 AS "Service Type",
        pickup_borough               AS "Pickup Borough",
        trip_count                   AS "Trip Count",
        total_revenue                AS "Total Revenue",
        total_fare_amount            AS "Total Fare Amount",
        total_tip_amount             AS "Total Tip Amount",
        avg_revenue_per_trip         AS "Avg Revenue Per Trip",
        avg_fare_amount              AS "Avg Fare Amount",
        avg_tip_percentage           AS "Avg Tip Percentage",
        avg_revenue_per_mile         AS "Avg Revenue Per Mile",
        avg_trip_distance            AS "Avg Trip Distance",
        avg_trip_duration_minutes    AS "Avg Trip Duration Minutes",
        revenue_per_trip             AS "Revenue Per Trip"
    FROM fct_daily_stats
    ORDER BY trip_date, service_type, pickup_borough
)
TO '{(OUTPUT_DIR / "daily_borough_stats_2025.csv").as_posix()}'
(HEADER, DELIMITER ',')
""")

# 2) Zone performance
con.execute(f"""
COPY (
    SELECT
        service_type                 AS "Service Type",
        location_id                  AS "Location ID",
        borough                      AS "Borough",
        zone_name                    AS "Zone Name",
        service_zone                 AS "Service Zone",
        total_trips                  AS "Total Trips",
        total_revenue                AS "Total Revenue",
        avg_revenue_per_trip         AS "Avg Revenue Per Trip",
        avg_revenue_per_mile         AS "Avg Revenue Per Mile",
        avg_tip_percentage           AS "Avg Tip Percentage",
        avg_trip_distance            AS "Avg Trip Distance",
        avg_trip_duration_minutes    AS "Avg Trip Duration Minutes",
        peak_pickup_hour             AS "Peak Pickup Hour",
        peak_hour_trip_count         AS "Peak Hour Trip Count",
        zone_efficiency_score        AS "Zone Efficiency Score",
        zone_efficiency_rank         AS "Zone Efficiency Rank"
    FROM dim_zones
    ORDER BY service_type, zone_efficiency_rank, zone_name
)
TO '{(OUTPUT_DIR / "zone_performance_2025.csv").as_posix()}'
(HEADER, DELIMITER ',')
""")

# 3) Time pattern / surge behavior
con.execute(f"""
COPY (
    SELECT
        service_type                 AS "Service Type",
        pickup_day_of_week_num       AS "Pickup Day Of Week Num",
        pickup_day_name              AS "Pickup Day Name",
        pickup_hour                  AS "Pickup Hour",
        time_of_day_bucket           AS "Time Of Day Bucket",
        trip_count                   AS "Trip Count",
        avg_fare_amount              AS "Avg Fare Amount",
        avg_total_amount             AS "Avg Total Amount",
        avg_tip_percentage           AS "Avg Tip Percentage",
        avg_revenue_per_mile         AS "Avg Revenue Per Mile",
        avg_trip_distance            AS "Avg Trip Distance",
        avg_trip_duration_minutes    AS "Avg Trip Duration Minutes",
        fare_index                   AS "Fare Index",
        total_amount_index           AS "Total Amount Index",
        fare_pattern_bucket          AS "Fare Pattern Bucket"
    FROM dim_time
    ORDER BY service_type, pickup_day_of_week_num, pickup_hour
)
TO '{(OUTPUT_DIR / "time_patterns_2025.csv").as_posix()}'
(HEADER, DELIMITER ',')
""")

# 4) Trip-level sample for BI demos
con.execute(f"""
COPY (
    SELECT
        service_type                 AS "Service Type",
        vendor_id                    AS "Vendor ID",
        pickup_datetime              AS "Pickup Datetime",
        dropoff_datetime             AS "Dropoff Datetime",
        trip_date                    AS "Trip Date",
        trip_year                    AS "Trip Year",
        trip_month                   AS "Trip Month",
        trip_day                     AS "Trip Day",
        pickup_hour                  AS "Pickup Hour",
        pickup_day_of_week_num       AS "Pickup Day Of Week Num",
        pickup_day_name              AS "Pickup Day Name",
        time_of_day_bucket           AS "Time Of Day Bucket",
        passenger_count              AS "Passenger Count",
        trip_distance                AS "Trip Distance",
        rate_code_id                 AS "Rate Code ID",
        pickup_location_id           AS "Pickup Location ID",
        pickup_borough               AS "Pickup Borough",
        pickup_zone_name             AS "Pickup Zone Name",
        pickup_service_zone          AS "Pickup Service Zone",
        dropoff_location_id          AS "Dropoff Location ID",
        dropoff_borough              AS "Dropoff Borough",
        dropoff_zone_name            AS "Dropoff Zone Name",
        dropoff_service_zone         AS "Dropoff Service Zone",
        payment_type                 AS "Payment Type",
        fare_amount                  AS "Fare Amount",
        extra                        AS "Extra",
        mta_tax                      AS "MTA Tax",
        tip_amount                   AS "Tip Amount",
        tolls_amount                 AS "Tolls Amount",
        improvement_surcharge        AS "Improvement Surcharge",
        congestion_surcharge         AS "Congestion Surcharge",
        cbd_congestion_fee           AS "CBD Congestion Fee",
        airport_fee                  AS "Airport Fee",
        ehail_fee                    AS "eHail Fee",
        trip_type                    AS "Trip Type",
        total_amount                 AS "Total Amount",
        trip_duration_minutes        AS "Trip Duration Minutes",
        revenue_per_mile             AS "Revenue Per Mile",
        tip_percentage               AS "Tip Percentage",
        average_speed_mph            AS "Average Speed MPH"
    FROM fct_trips
    ORDER BY pickup_datetime
    LIMIT 100000
)
TO '{(OUTPUT_DIR / "trip_level_sample_2025.csv").as_posix()}'
(HEADER, DELIMITER ',')
""")

print("Exported files:")
for f in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", f.name)


con.close()

CSV files exported to: /Users/girimanoharv/Documents/Taxi-Revenue-and-Surge-Analytics/outputs/csv_exports
Exported files:
- daily_borough_stats_2025.csv
- dim_time.csv
- dim_zones.csv
- fct_daily_stats.csv
- fct_trips_sample.csv
- time_patterns_2025.csv
- trip_level_sample_2025.csv
- zone_performance_2025.csv
